# NHS Wales Emergency Department Performance
## Notebook 1: load, check and clean the data

**Question:** How are Welsh health boards performing against the 4-hour and 12-hour emergency department targets, and how reliable is the data?

**Source:** StatsWales, *Emergency departments: performance against waiting times targets, October 2009 onwards* (accredited official statistics, Digital Health and Care Wales).
https://stats.gov.wales/en-GB/3ce1cc28-0625-4674-89d6-9cf3a56f2d9c/start

**Targets in Wales:** 95% of patients admitted, transferred or discharged within 4 hours, and no one waiting 12 hours or more.

Run each cell in order with Shift + Enter. Read the output before moving on: every check is written down so the work is repeatable.

In [ ]:
from pathlib import Path
import pandas as pd
import plotly.express as px

PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = PROJECT / "data" / "raw"
OUT = PROJECT / "data" / "processed"
OUT.mkdir(parents=True, exist_ok=True)

files = [f for f in RAW.glob("*.csv") if not f.name.startswith("test_")]
if not files:
    files = list(RAW.glob("test_*.csv"))
    print("No StatsWales download found, using the TEST file. Download the real CSV into data/raw.")
RAW_FILE = max(files, key=lambda f: f.stat().st_mtime)
print("Reading:", RAW_FILE.name)

## 1. Load and inspect the raw file
Never edit the raw file. All changes happen in code.

In [ ]:
raw = pd.read_csv(RAW_FILE)
print(raw.shape[0], "rows,", raw.shape[1], "columns")
raw.head()

In [ ]:
# The columns we need, renamed to short names
df = raw.rename(columns={
    "Data values": "value",
    "Data description": "measure",
    "Date_reference": "month_ref",
    "Hospital": "area",
    "Hospital_reference": "area_code",
    "Hospital_hierarchy": "parent_code",
    "Hospital Type_reference": "unit_type",
    "Target": "target",
})[["value", "measure", "month_ref", "area", "area_code", "parent_code", "unit_type", "target"]]

df["parent_code"] = df["parent_code"].fillna("")
df["month"] = pd.to_datetime(df["month_ref"].str.replace("m", "-") + "-01")
df["measure"] = df["measure"].map({
    "Total attendances": "total",
    "Attendances where patients spend less than the target time in an emergency department": "within",
    "Percentage of patients who spend less than the target time in an emergency department": "pct",
})
df["level"] = df.apply(lambda r: "Wales" if r.area == "Wales"
                       else ("Health board" if r.parent_code == "XXXW92000004" else "Hospital"), axis=1)
df.head()

## 2. Data quality checks
Each check adds a line to the data quality log, which is saved at the end and goes on the dashboard.

In [ ]:
dq_log = []
def log(check, finding, action):
    dq_log.append({"check": check, "finding": finding, "action": action})
    print(f"[{check}] {finding}  ->  {action}")

In [ ]:
# 2a. Missing and non-numeric values
missing = df.isna().sum()
missing = missing[missing > 0]
non_numeric = pd.to_numeric(df["value"], errors="coerce").isna().sum()
df["value"] = pd.to_numeric(df["value"], errors="coerce")
log("Missing values",
    "None" if missing.empty else missing.to_dict(),
    "No action needed" if missing.empty else "Investigate before analysis")
log("Non-numeric values", f"{non_numeric} found",
    "No action needed" if non_numeric == 0 else "Converted to blank and excluded")

In [ ]:
# 2b. Duplicate rows (same month, area, unit type, measure and target)
key = ["month_ref", "area_code", "unit_type", "measure", "target"]
dupes = df.duplicated(key).sum()
log("Duplicates", f"{dupes} duplicate rows", "No action needed" if dupes == 0 else "Kept first, dropped the rest")
df = df.drop_duplicates(key)

In [ ]:
# 2c. Recalculate each published percentage from the counts and compare
totals = df[df.measure == "total"].set_index(["month_ref", "area_code", "unit_type"])["value"]
within = df[df.measure == "within"].set_index(["month_ref", "area_code", "unit_type", "target"])["value"]
pct = df[df.measure == "pct"].set_index(["month_ref", "area_code", "unit_type", "target"])["value"]

check = pd.DataFrame({"within": within, "published_pct": pct}).reset_index()
check = check.merge(totals.rename("total").reset_index(), on=["month_ref", "area_code", "unit_type"])
check["recalc_pct"] = 100 * check["within"] / check["total"]
check["diff"] = (check["recalc_pct"] - check["published_pct"]).abs()
bad = check[check["diff"] > 0.05]
log("Recalculated percentages",
    f"{len(check):,} percentages checked, {len(bad)} differ by more than 0.05 points (largest gap {check['diff'].max():.5f})",
    "Use counts and recalculate percentages myself" if len(bad) == 0 else "Investigate mismatches")

In [ ]:
# 2d. Areas that start or stop part-way through: renames, openings, closures, changes of unit type
span = df.groupby(["area", "unit_type"])["month"].agg(first="min", last="max").reset_index()
latest = df["month"].max()
earliest = df["month"].min()
changes = span[(span["first"] > earliest) | (span["last"] < latest)].sort_values("first")
changes

In [ ]:
for _, r in changes.iterrows():
    if r["last"] < latest:
        log("Series ends", f"{r.area} ({r.unit_type}) stops after {r['last']:%b %Y}",
            "Kept history, excluded from current comparisons")
    if r["first"] > earliest:
        log("Series starts", f"{r.area} ({r.unit_type}) starts {r['first']:%b %Y}",
            "Trend charts for this area start from this date")

In [ ]:
# 2e. Health boards with no major emergency department
boards = df[df.level == "Health board"]
board_types = boards.groupby("area")["unit_type"].unique()
no_major = [b for b, t in board_types.items() if "Major" not in t]
for b in no_major:
    log("No major ED", f"{b} only has minor injury units, so its figures are not comparable with other boards",
        "Excluded from board rankings, shown separately")

In [ ]:
# 2f. Do the health boards add up to the Wales total?
tot_all = df[(df.measure == "total") & (df.unit_type == "All")]
wales = tot_all[tot_all.level == "Wales"].set_index("month_ref")["value"]
sum_boards = tot_all[tot_all.level == "Health board"].groupby("month_ref")["value"].sum()
gap = (wales - sum_boards).abs()
log("Boards add up to Wales", f"Largest monthly gap: {gap.max():,.0f} attendances",
    "No action needed" if gap.max() == 0 else "Investigate the months with gaps")

In [ ]:
# 2g. When does each target's data begin?
for t, d in df[df.measure == "within"].groupby("target")["month"].min().items():
    log("Target start date", f"{t} data starts {d:%b %Y}", "Charts for this target start here")

## 3. Build a tidy table for analysis
One row per month, area and unit type, with counts and recalculated percentages.

In [ ]:
wide = df[df.measure.isin(["total", "within"])].copy()
wide["col"] = wide.apply(lambda r: "total" if r.measure == "total"
                         else "within_" + r.target.split()[0] + "h", axis=1)
tidy = (wide.pivot_table(index=["month", "area", "area_code", "parent_code", "level", "unit_type"],
                         columns="col", values="value", aggfunc="first")
            .reset_index())
tidy.columns.name = None
tidy["pct_4h"] = 100 * tidy["within_4h"] / tidy["total"]
tidy["pct_12h_or_less"] = 100 * tidy["within_12h"] / tidy["total"]
tidy["over_12h"] = tidy["total"] - tidy["within_12h"]
tidy = tidy.sort_values(["month", "level", "area"])
tidy.tail()

In [ ]:
tidy.to_csv(OUT / "ed_performance_tidy.csv", index=False)
pd.DataFrame(dq_log).to_csv(OUT / "data_quality_log.csv", index=False)
print("Saved:", OUT / "ed_performance_tidy.csv")
print("Saved:", OUT / "data_quality_log.csv")
pd.DataFrame(dq_log)

## 4. First look at the data

In [ ]:
wales = tidy[(tidy.level == "Wales") & (tidy.unit_type == "All") & (tidy.month >= "2019-04-01")]
fig = px.line(wales, x="month", y="pct_4h",
              title="Wales: % of patients seen within 4 hours, all emergency care facilities",
              labels={"month": "", "pct_4h": "% within 4 hours"})
fig.add_hline(y=95, line_dash="dash", annotation_text="95% target")
fig.show()

In [ ]:
last12 = tidy[(tidy.month > tidy.month.max() - pd.DateOffset(months=12)) & (tidy.unit_type == "All")
              & (tidy.level == "Health board") & (~tidy.area.isin(no_major))]
rank = (last12.groupby("area")[["total", "within_4h", "within_12h"]].sum()
              .assign(pct_4h=lambda d: 100 * d.within_4h / d.total,
                      pct_over_12h=lambda d: 100 * (d.total - d.within_12h) / d.total)
              .sort_values("pct_4h"))
fig = px.bar(rank.reset_index(), x="pct_4h", y="area", orientation="h",
             title="Health boards: % within 4 hours, last 12 months",
             labels={"pct_4h": "% within 4 hours", "area": ""}, text_auto=".1f")
fig.add_vline(x=95, line_dash="dash", annotation_text="95% target")
fig.show()
rank.round(1)

## 5. Findings

1. **Performance is getting worse, not better.** In October 2026, 61.1% of patients in Wales spent less than 4 hours in an emergency department. That is the lowest month since the health boards were reorganised in April 2019, down from 66.0% a year earlier and 75.3% in October 2019. The target is 95%.

2. **12-hour waits are rising.** 131,500 patients waited 12 hours or more in the last 12 months, up 8% on the year before (121,492). In October 2026 alone it was 11,830 patients, more than double October 2019 (5,580).

3. **Board figures hide large differences between hospitals.** Aneurin Bevan has the best board figure at 72.5% within 4 hours, but its major ED, The Grange, managed only 45.6%. The board total is lifted by its minor injury units. The widest gap is between hospitals: at Ysbyty Glan Clwyd 33.2% were seen within 4 hours and 29.0% waited 12 hours or more, while Bronglais reached 64.8%. Comparing boards without splitting major and minor units would give a misleading picture.

**Data quality:** all 26,736 published percentages match a recalculation from the counts, health boards add up exactly to the Wales total, and the main risks to comparison are the 2019 board changes, The Grange opening in November 2020, and Powys having no major ED.